# Kafka → MinIO Streaming with Spark

Читает данные из Kafka топика и пишет в MinIO (S3) в формате Parquet.

## Преимущества:
- Контроль размера файлов (объединяет все 72 партиции Kafka)
- Нет проблемы с мелкими файлами
- Автоматическая обработка в реальном времени
- Можно делать трансформации на лету

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.avro.functions import from_avro
from pyspark.sql.types import *
import json

# Создание Spark сессии с поддержкой Kafka и S3
spark = SparkSession.builder \
    .appName("Kafka-to-MinIO-Streaming") \
    .config("spark.sql.streaming.checkpointLocation", "/tmp/checkpoints/kafka-to-minio") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark UI: http://localhost:4040")

## Конфигурация

In [ ]:
# Kafka настройки
KAFKA_BOOTSTRAP_SERVERS = "kafka:9092"
KAFKA_TOPIC = "order-events"
SCHEMA_REGISTRY_URL = "http://schema-registry:8081"

# MinIO/S3 настройки
OUTPUT_BUCKET = "datalake"
OUTPUT_PATH = "topics-streaming/order-events"
CHECKPOINT_LOCATION = "/tmp/checkpoints/kafka-to-minio"

# Настройки записи
TRIGGER_INTERVAL = "5 minutes"  # Как часто записывать данные
NUM_OUTPUT_FILES = 4  # Количество файлов на каждую запись

print(f"Kafka: {KAFKA_BOOTSTRAP_SERVERS} / {KAFKA_TOPIC}")
print(f"Output: s3a://{OUTPUT_BUCKET}/{OUTPUT_PATH}/")
print(f"Trigger: every {TRIGGER_INTERVAL}")
print(f"Files per batch: {NUM_OUTPUT_FILES}")

## Получение Avro схемы из Schema Registry

In [ ]:
import requests

def get_avro_schema_from_registry(subject_name):
    """
    Получает Avro схему из Confluent Schema Registry
    """
    url = f"{SCHEMA_REGISTRY_URL}/subjects/{subject_name}/versions/latest"
    response = requests.get(url)
    
    if response.status_code == 200:
        schema_json = response.json()['schema']
        return schema_json
    else:
        raise Exception(f"Failed to get schema: {response.status_code} - {response.text}")

# Получаем схему для топика order-events
avro_schema = get_avro_schema_from_registry(f"{KAFKA_TOPIC}-value")
print("Avro schema loaded:")
print(json.dumps(json.loads(avro_schema), indent=2)[:500] + "...")

## Чтение из Kafka

In [ ]:
# Читаем из Kafka
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "earliest") \
    .option("maxOffsetsPerTrigger", 100000) \
    .load()

print("Kafka stream schema:")
kafka_df.printSchema()

## Десериализация Avro (без Schema Registry ID)

Kafka Connect записывает в Avro с Schema Registry ID в начале (5 байт).
Нужно их пропустить.

In [ ]:
# Пропускаем первые 5 байт (magic byte + schema ID)
# и десериализуем Avro
df = kafka_df.select(
    from_avro(
        F.expr("substring(value, 6, length(value)-5)"),  # Пропускаем первые 5 байт
        avro_schema
    ).alias("data"),
    F.col("timestamp").alias("kafka_timestamp"),
    F.col("partition").alias("kafka_partition"),
    F.col("offset").alias("kafka_offset")
)

# Раскрываем структуру
df = df.select("data.*", "kafka_timestamp", "kafka_partition", "kafka_offset")

# Добавляем партиционные поля (calc_id, dt, hour)
df = df.withColumn("calc_id", F.date_format(F.col("kafka_timestamp"), "yyyyMMdd-HHmmss")) \
       .withColumn("dt", F.date_format(F.col("kafka_timestamp"), "yyyy-MM-dd")) \
       .withColumn("hour", F.date_format(F.col("kafka_timestamp"), "HH"))

print("Deserialized schema:")
df.printSchema()

## Запись в MinIO (S3)

### Ключевые параметры:
- **trigger**: Как часто записывать (каждые 5 минут)
- **coalesce**: Объединяет все 72 Kafka партиции в N файлов
- **partitionBy**: Партиционирование по времени
- **format**: Parquet с Snappy компрессией

In [ ]:
def write_batch(batch_df, batch_id):
    """
    Функция для записи каждого батча.
    Здесь можно контролировать количество файлов.
    """
    print(f"\n{'='*70}")
    print(f"Processing batch {batch_id}")
    print(f"{'='*70}")
    
    row_count = batch_df.count()
    print(f"Rows in batch: {row_count:,}")
    
    if row_count > 0:
        # Уменьшаем количество партиций для объединения файлов
        # 72 Kafka партиции → NUM_OUTPUT_FILES файлов
        batch_df_coalesced = batch_df.coalesce(NUM_OUTPUT_FILES)
        
        # Записываем в Parquet
        batch_df_coalesced.write \
            .mode("append") \
            .format("parquet") \
            .option("compression", "snappy") \
            .option("parquet.block.size", 268435456) \
            .option("parquet.page.size", 1048576) \
            .partitionBy("calc_id", "dt", "hour") \
            .save(f"s3a://{OUTPUT_BUCKET}/{OUTPUT_PATH}/")
        
        print(f"Written {row_count:,} rows in {NUM_OUTPUT_FILES} files")
    else:
        print("Empty batch, skipping")
    
    print(f"{'='*70}\n")

print("Batch writer function defined")

In [ ]:
# Запускаем streaming query
query = df.writeStream \
    .foreachBatch(write_batch) \
    .trigger(processingTime=TRIGGER_INTERVAL) \
    .option("checkpointLocation", CHECKPOINT_LOCATION) \
    .start()

print(f"Streaming started!")
print(f"Query ID: {query.id}")
print(f"Status: {query.status}")
print(f"\nTo stop: query.stop()")

## Мониторинг

In [ ]:
# Проверка статуса
print(f"Query ID: {query.id}")
print(f"Status: {query.status}")
print(f"Recent progress:")
print(query.lastProgress)

In [ ]:
# Ждем завершения (или Ctrl+C для остановки)
# query.awaitTermination()

# Или ждем 1 час и останавливаем
import time
try:
    print("Running for 1 hour... Press Ctrl+C to stop earlier")
    time.sleep(3600)
except KeyboardInterrupt:
    print("\nStopping...")
finally:
    query.stop()
    print("Stopped")

## Проверка результатов

In [ ]:
import s3fs

s3 = s3fs.S3FileSystem(
    key='minioadmin',
    secret='minioadmin',
    client_kwargs={'endpoint_url': 'http://minio:9000'}
)

# Файлы в MinIO
output_files = s3.glob(f'{OUTPUT_BUCKET}/{OUTPUT_PATH}/**/*.parquet')
print(f"\nOutput files: {len(output_files)}")

if output_files:
    total_size_mb = sum([s3.size(f) for f in output_files]) / (1024 * 1024)
    print(f"Total size: {total_size_mb:.2f} MB")
    print(f"Average file size: {total_size_mb / len(output_files):.2f} MB")
    
    print("\nRecent files:")
    for f in sorted(output_files)[-10:]:
        size_mb = s3.size(f) / (1024 * 1024)
        print(f"  {f}: {size_mb:.2f} MB")
else:
    print("No files yet")

In [ ]:
# Читаем записанные данные для проверки
result_df = spark.read.parquet(f"s3a://{OUTPUT_BUCKET}/{OUTPUT_PATH}/")

print(f"Total rows: {result_df.count():,}")
print("\nSchema:")
result_df.printSchema()
print("\nSample:")
result_df.show(5, truncate=False)

## Production deployment

### Вариант 1: Docker container
```dockerfile
FROM apache/spark-py:latest
COPY kafka-to-minio.py /app/
CMD ["/opt/spark/bin/spark-submit", "/app/kafka-to-minio.py"]
```

### Вариант 2: Kubernetes (Spark Operator)
```yaml
apiVersion: sparkoperator.k8s.io/v1beta2
kind: SparkApplication
metadata:
  name: kafka-to-minio
spec:
  type: Python
  mode: cluster
  image: your-spark-image:latest
  mainApplicationFile: local:///app/kafka-to-minio.py
```

### Вариант 3: Docker Compose (добавить в ваш compose.yaml)
```yaml
spark-streaming:
  image: apache/spark-py:3.5.0
  command: |
    /opt/spark/bin/spark-submit \
      --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,org.apache.spark:spark-avro_2.12:3.5.0 \
      /notebooks/kafka-to-minio.py
  volumes:
    - ./jupyter/notebooks:/notebooks
```